# Validierung der Wahl des Schwellwerts für untervertretene Klassen (Leakage-Check)

In [1]:
from pathlib import Path
import pandas as pd
import sys

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS
from models_helper import per_class_learning_curve
from sklearn.ensemble import RandomForestClassifier
from _settings import SEED

In [2]:
DATA_DIR = Path("../3_9_split dataset to datasets and remove rare classes")

df_train = pd.read_parquet(DATA_DIR / "dataset_train_rare_classes_removed.parquet")
df_val = pd.read_parquet(DATA_DIR / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


## 1. Class distribution analysis

In [3]:
# show main labels
LABEL_COLS = [c for c in df_train.columns if c.startswith("label_")]

# show class distribution for each label column
for col in LABEL_COLS:
    counts = df_train[col].value_counts()
    print(f"\n{col} ({len(counts)} classes):")
    print(counts.to_string())


label_ifc_entity (13 classes):
label_ifc_entity
IfcWall                12314
IfcRailing              6938
IfcSpace                2830
IfcWindow               1174
IfcSlab                  920
IfcDoor                  731
IfcRoof                  456
IfcColumn                454
IfcPlate                 404
IfcCovering              330
IfcSanitaryTerminal      329
IfcStairFlight            95
IfcFurniture               2

label_predefined_type (17 classes):
label_predefined_type
SOLIDWALL        7359
GUARDRAIL        6938
NOTDEFINED       3637
INTERNAL         2704
PLUMBINGWALL     1399
WINDOW           1174
DOOR              731
FLOOR             659
PARTITIONING      480
FLAT_ROOF         456
SHEET             404
FLOORING          330
COLUMN            300
BASESLAB          261
GFA               126
WASHHANDBASIN      11
TOILETPAN           8

label_is_external (3 classes):
label_is_external
true       14329
false       9391
unknown     3257

label_load_bearing (3 classes):
label_l

## 2. Per-class learning curve analysis

In [4]:
# create feature matrices
X_train = df_train[ALL_FEATURE_KEYS].values
X_val = df_val[ALL_FEATURE_KEYS].values

# baseline model for learning curve analysis, RF can handle small sample sizes
model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

# set candidate threshold for rare classes
CANDIDATE_THRESHOLD = 1000

# analyze each label column for rare classes and plot learning curves
for col in LABEL_COLS:
    y_train = df_train[col].values
    y_val = df_val[col].values
    counts = df_train[col].value_counts()
    rare_candidates = counts[counts < CANDIDATE_THRESHOLD].index.tolist()

    if not rare_candidates:
        print(f"{col}: No rare classes detected")
        continue

    print(f"\n{col} - {len(rare_candidates)} rare candidates:")
    for cls in rare_candidates:
        print(f"  '{cls}': {(y_train == cls).sum()} Train / {(y_val == cls).sum()} Val Samples")

    per_class_learning_curve(
        X_train, y_train, model,
        class_labels=rare_candidates,
        label_name=col,
        X_val=X_val, y_val=y_val,
        threshold=200,
        save=f"learning_curve_validated_{col}",
        chapter="implementation"
    )


label_ifc_entity - 9 rare candidates:
  'IfcSlab': 920 Train / 805 Val Samples
  'IfcDoor': 731 Train / 392 Val Samples
  'IfcRoof': 456 Train / 308 Val Samples
  'IfcColumn': 454 Train / 73 Val Samples
  'IfcPlate': 404 Train / 732 Val Samples
  'IfcCovering': 330 Train / 37 Val Samples
  'IfcSanitaryTerminal': 329 Train / 555 Val Samples
  'IfcStairFlight': 95 Train / 35 Val Samples
  'IfcFurniture': 2 Train / 68 Val Samples
figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/learning_curve_validated_label_ifc_entity.svg



label_predefined_type - 11 rare candidates:
  'DOOR': 731 Train / 392 Val Samples
  'FLOOR': 659 Train / 774 Val Samples
  'PARTITIONING': 480 Train / 651 Val Samples
  'FLAT_ROOF': 456 Train / 308 Val Samples
  'SHEET': 404 Train / 732 Val Samples
  'FLOORING': 330 Train / 37 Val Samples
  'COLUMN': 300 Train / 63 Val Samples
  'BASESLAB': 261 Train / 31 Val Samples
  'GFA': 126 Train / 136 Val Samples
  'WASHHANDBASIN': 11 Train / 319 Val Samples
  'TOILETPAN': 8 Train / 236 Val Samples
figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/learning_curve_validated_label_predefined_type.svg


label_is_external: No rare classes detected
label_load_bearing: No rare classes detected
